<a href="https://colab.research.google.com/github/peter-adepoju/math-neuro-ml/blob/main/week6/lesson20/Lesson20_Exercises_Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Lesson 20 — Exercises**
## Inference and Model Checking: Standard Errors, Bootstrap, Correlation and ROC

**Course:** MAMBA: Mathematical Sciences for Brain Data\
**Week:** Week 6 — Differential Equations\
**Lesson:** Lesson 20

---
This notebook has **7 required exercises worth 100 points total**, plus one **bonus
challenge** (extra credit, not counted in the 100). Work through them in order — later
exercises reuse functions and data from earlier ones.

**How to use this notebook:**
- Cells marked `# YOUR CODE HERE` are where you write code. Remove the `raise
  NotImplementedError` line when you add your answer.
- Cells marked **Written response** are where you type a short answer in the Markdown cell
  provided (double-click the cell to edit it).
- Cells marked **Self-check** run lightweight checks on your work. Passing a self-check is a
  good sign, but it does not guarantee full credit — it cannot check written reasoning.
- Do not change the random seed or provided data unless an exercise explicitly asks you to.

**Point allocation**

| Exercise | Topic | Points |
|---|---|---|
| 1 | Bias, standard error and mean squared error | 15 |
| 2 | Confidence intervals and coverage | 15 |
| 3 | The bootstrap | 15 |
| 4 | Correlation, nonlinearity and confounding | 15 |
| 5 | ROC curves and AUC | 15 |
| 6 | Dispersion checks with error bars | 10 |
| 7 | Real data: error bars for a second mouse | 15 |
| **Total** | | **100** |
| Bonus | A permutation test for an AUC | extra credit |

In [ ]:
# SETUP CELL -- run this first.
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

PALETTE = ['#0072B2', '#D55E00', '#009E73', '#CC79A7', '#E69F00', '#56B4E9']

print("Setup complete.")

---
## Exercise 1 — Bias, Standard Error and Mean Squared Error (15 points)

The true Fano factor of a Poisson neuron is 1. With few trials, how good are different estimators of it, and of the variance?

In [ ]:
# 1.1 (6 points): With rng1, simulate 20,000 experiments, each with n = 8 trials of Poisson counts with mean 3 (array (20000, 8)).
# For each experiment compute: var_n (divisor n), var_n1 (divisor n - 1), and ff = var_n1 / mean (skip experiments with mean 0).
# Store, for var_n and var_n1, their bias (true variance 3), SE and RMSE in dict res1 = {'var_n': (bias, se, rmse), 'var_n1': (...)},
# and the bias and SE of the Fano-factor estimator in ff_bias1, ff_se1.

rng1 = np.random.default_rng(1)

# YOUR CODE HERE
res1 = ff_bias1 = ff_se1 = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 1.1
assert res1 is not None and ff_bias1 is not None and ff_se1 is not None, "Set res1, ff_bias1 and ff_se1."
assert abs(res1['var_n'][0] + 3 / 8) < 0.05 and abs(res1['var_n1'][0]) < 0.05, "Divisor n is biased by -sigma^2/n; divisor n-1 is unbiased."
assert res1['var_n'][2] < res1['var_n1'][2], "Here the biased estimator has the smaller RMSE (smaller SE)."
assert abs(ff_bias1) < 0.05 and 0.45 < ff_se1 < 0.6, "The Fano factor from 8 trials is nearly unbiased but very noisy."
print("Self-check passed.")

In [ ]:
# 1.2 (5 points): The proportion correct p_hat = k/n has SE sqrt(p(1-p)/n). (a) For p = 0.7, how many trials are needed for SE <= 0.05, and
# for SE <= 0.025? Store n_005 and n_0025 (smallest integers). (b) Store se_diff1 = the SE of the difference between two independent
# accuracies, 0.7 from 60 trials and 0.75 from 60 trials.

# YOUR CODE HERE
n_005 = n_0025 = se_diff1 = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 1.2
assert all(v is not None for v in (n_005, n_0025, se_diff1)), "Set n_005, n_0025 and se_diff1."
assert n_005 == 84 and n_0025 == 336 and np.isclose(se_diff1, np.sqrt(0.21 / 60 + 0.1875 / 60)), "n = p(1-p)/SE^2; variances of independent estimates add."
print("Self-check passed.")

**1.3 (4 points, written response).** (a) In 1.1, the biased variance estimator has the smaller RMSE. How is that possible, and when would you still prefer the unbiased one? (b) A Fano factor of 1.6 is reported from 8 trials of one neuron. Using your SE, is that evidence of super-Poisson variability? (c) Explain the difference between the SD of the data and the SE of an estimate, with an example.

**Your answer:**

*(replace this line with your answer)*

---
## Exercise 2 — Confidence Intervals and Coverage (15 points)

In [ ]:
# 2.1 (6 points): Write wald_ci(k, n) and wilson_ci(k, n) (95%, z = 1.96), vectorised over k. With rng2 and 20,000 simulated experiments each,
# estimate the coverage of both intervals for n = 15 and true p in p_list2. Store cov_wald2 and cov_wilson2 (each shape (4,)).

rng2 = np.random.default_rng(2)
p_list2 = [0.5, 0.7, 0.9, 0.97]

# YOUR CODE HERE
cov_wald2 = cov_wilson2 = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 2.1
assert cov_wald2 is not None and cov_wilson2 is not None, "Set cov_wald2 and cov_wilson2."
assert cov_wald2[-1] < 0.65 and cov_wilson2[-1] > 0.85, "Near p = 1 the Wald interval fails; Wilson does much better."
lo_, hi_ = wilson_ci(np.array([15]), 15)
assert hi_[0] <= 1 and lo_[0] < 1, "The Wilson interval stays inside [0, 1] and has width even when k = n."
print("Self-check passed.")

In [ ]:
# 2.2 (5 points): Coverage depends on the data, not only on the formula. For the mean of n intervals drawn from a lognormal distribution
# (rng2.lognormal(mean=3, sigma=1, size=...)), whose true mean is exp(3 + 0.5), compute the coverage of mean +- 1.96 * s / sqrt(n) for
# n in [5, 20, 80, 320] (10,000 experiments each). Store cov_lognorm2 (shape (4,)).

# YOUR CODE HERE
cov_lognorm2 = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 2.2
assert cov_lognorm2 is not None and cov_lognorm2.shape == (4,), "Set cov_lognorm2."
assert cov_lognorm2[0] < 0.85 and cov_lognorm2[-1] > cov_lognorm2[0] and cov_lognorm2[-1] > 0.9, "Coverage improves with n but is poor for small n."
print("Self-check passed.")

**2.3 (4 points, written response).** (a) A student writes: "the 95% CI for the decoder's accuracy is [0.71, 0.89], so there is a 95% probability that the true accuracy is in [0.71, 0.89]". Correct the statement, and say under what extra assumptions a statement of that form is legitimate. (b) Why does the Wald interval fail near $p = 1$? (c) Why is coverage poor for small samples of lognormal intervals?

**Your answer:**

*(replace this line with your answer)*

---
## Exercise 3 — The Bootstrap (15 points)

In [ ]:
# 3.1 (6 points): Write bootstrap(data, stat, B, rng) that resamples ROWS (units) with replacement and returns the B values of stat.
# For the intervals isi3 below: store the bootstrap SE of the mean (se_boot3) and the analytic SE (se_analytic3); the 95% percentile
# interval for the median (ci_med3); and the 95% percentile interval for the CV (ci_cv3). Use B = 4000 and rng3.

rng3 = np.random.default_rng(3)
isi3 = np.random.default_rng(31).gamma(2, 15, 60)                            # (60,) interspike intervals, ms (gamma, CV 0.71)

# YOUR CODE HERE
se_boot3 = se_analytic3 = ci_med3 = ci_cv3 = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 3.1
assert all(v is not None for v in (se_boot3, se_analytic3, ci_med3, ci_cv3)), "Set all four results."
assert abs(se_boot3 / se_analytic3 - 1) < 0.08, "For the mean, bootstrap and analytic SEs agree."
assert ci_med3[0] < np.median(isi3) < ci_med3[1] and ci_cv3[0] < cv(isi3) < ci_cv3[1], "Intervals contain the estimates."
assert bootstrap(np.arange(5.0), np.mean, 50, np.random.default_rng(0)).shape == (50,), "bootstrap returns B values."
print("Self-check passed.")

In [ ]:
# 3.2 (5 points): What to resample matters. countsT3 below holds spike counts of one neuron on 40 trials x 10 consecutive 100 ms bins. Within a
# trial, bins share a trial-level gain (so they are correlated). Estimate the SE of the overall mean count per bin by bootstrapping
# (a) TRIALS (rows; correct here) and (b) individual BINS as if all 400 were independent (wrong). Store se_trials3 and se_bins3 (B = 4000).

countsT3 = np.random.default_rng(32).poisson(np.random.default_rng(33).gamma(4, 1.0, size=(40, 1)) * np.ones((1, 10)))    # (40, 10)

# YOUR CODE HERE
se_trials3 = se_bins3 = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 3.2
assert se_trials3 is not None and se_bins3 is not None, "Set se_trials3 and se_bins3."
assert se_trials3 > 1.8 * se_bins3, "Treating correlated bins as independent understates the SE."
print("Self-check passed.")

**3.3 (4 points, written response).** (a) Why does resampling bins understate the SE in 3.2, by roughly how much, and why? (b) In a study with 5 mice, 40 neurons per mouse and 100 trials per neuron, what would you resample to put an interval on "the mean effect in the population of mice"? (c) Name one situation where the bootstrap percentile interval is unreliable.

**Your answer:**

*(replace this line with your answer)*

---
## Exercise 4 — Correlation, Nonlinearity and Confounding (15 points)

In [ ]:
# 4.1 (6 points): (a) Write pearson(x, y) (mean product of z-scores) and check it against np.corrcoef. (b) For the tuning data (ori4, rate4),
# store the Pearson r (r_lin4) and the Pearson r between rate4 and cos(2 * radians(ori4 - 90)) (r_cos4). (c) For data with one outlier
# (xo4, yo4), store Pearson (r_out4) and Spearman (rho_out4).

rng4 = np.random.default_rng(4)
ori4 = rng4.uniform(0, 180, 150)                                              # (150,) orientation, degrees
rate4 = 10 + 8 * np.cos(2 * np.radians(ori4 - 90)) + rng4.normal(0, 2, 150)   # (150,) spikes/s, preferred 90 deg
xo4 = np.r_[rng4.normal(0, 1, 25), 7.0]; yo4 = np.r_[rng4.normal(0, 1, 25), -7.0]   # (26,) each

# YOUR CODE HERE
r_lin4 = r_cos4 = r_out4 = rho_out4 = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 4.1
assert all(v is not None for v in (r_lin4, r_cos4, r_out4, rho_out4)), "Set all four results."
assert abs(r_lin4) < 0.3 and r_cos4 > 0.85, "Strong but nonlinear: only a weak linear correlation with orientation."
assert r_out4 < -0.5 and abs(rho_out4) < abs(r_out4), "One outlier dominates Pearson, much less Spearman."
print("Self-check passed.")

In [ ]:
# 4.2 (5 points): Two neurons both depend on the animal's running speed (speed4), but not on each other. Store the pooled correlation of their
# rates (r_pool4), the partial correlation after regressing each on speed (r_partial4; use np.polyfit(speed4, y, 1) residuals), and a 95%
# bootstrap interval for the partial correlation, resampling TRIALS (ci_partial4; B = 2000, rng4; recompute the regressions inside each resample).

speed4 = rng4.gamma(2, 5, 300)                                               # (300,) cm/s
nA4 = 5 + 0.8 * speed4 + rng4.normal(0, 3, 300)                              # (300,) spikes/s
nB4 = 2 + 0.5 * speed4 + rng4.normal(0, 3, 300)

# YOUR CODE HERE
r_pool4 = r_partial4 = ci_partial4 = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 4.2
assert all(v is not None for v in (r_pool4, r_partial4, ci_partial4)), "Set r_pool4, r_partial4 and ci_partial4."
assert r_pool4 > 0.5 and abs(r_partial4) < 0.15 and ci_partial4[0] < 0 < ci_partial4[1], "The correlation is explained by speed."
print("Self-check passed.")

**4.3 (4 points, written response).** (a) Why is the correlation between rate and orientation near zero although orientation controls the rate? (b) In 4.2, what would you conclude if the partial correlation were still 0.3 with an interval excluding 0, and what could still be wrong? (c) Why must the bootstrap in 4.2 resample whole trials, and recompute the regression each time?

**Your answer:**

*(replace this line with your answer)*

---
## Exercise 5 — ROC Curves and AUC (15 points)

In [ ]:
# 5.1 (6 points): Write roc_points(score, label) returning arrays (fa, hit) for the thresholds 'score >= t' at +inf and every distinct score
# (descending), and auc_mw(score, label) = P(s1 > s0) + 0.5 P(s1 == s0). For the decoder scores sc5 and labels lab5 below, store
# auc_trap5 (trapezoid area under your ROC points) and auc_mw5, and check they agree.

rng5 = np.random.default_rng(5)
lab5 = rng5.random(300) < 0.4                                                 # (300,) truth
sc5 = rng5.poisson(np.where(lab5, 7.0, 4.0))                                  # (300,) spike-count scores

# YOUR CODE HERE
auc_trap5 = auc_mw5 = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 5.1
assert auc_trap5 is not None and auc_mw5 is not None, "Set auc_trap5 and auc_mw5."
assert np.isclose(auc_trap5, auc_mw5) and 0.75 < auc_mw5 < 0.9, "Trapezoid area = Mann-Whitney probability (ties as 1/2)."
assert fa5[0] == 0 and hit5[0] == 0 and fa5[-1] == 1 and hit5[-1] == 1, "The ROC runs from (0, 0) to (1, 1)."
print("Self-check passed.")

In [ ]:
# 5.2 (5 points): Properties of AUC. (a) Store auc_log5 = AUC of log(1 + sc5) and auc_neg5 = AUC of -sc5.
# (b) Turn the counts into "probabilities" with two different calibrations: q_good = expit(0.6 * (sc5 - 5.5)) and q_bad = expit(5 * (sc5 - 5.5));
#     store auc_good5, auc_bad5 and their log losses ll_good5, ll_bad5.
# (c) Store boot_ci5, the 95% bootstrap interval of the AUC resampling trials (B = 2000, rng5).

from scipy.special import expit

# YOUR CODE HERE
auc_log5 = auc_neg5 = auc_good5 = auc_bad5 = ll_good5 = ll_bad5 = boot_ci5 = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 5.2
assert all(v is not None for v in (auc_log5, auc_neg5, auc_good5, auc_bad5, ll_good5, ll_bad5, boot_ci5)), "Set all results."
assert np.isclose(auc_log5, auc_mw5) and np.isclose(auc_neg5, 1 - auc_mw5), "AUC is invariant to increasing transforms; reversing gives 1 - AUC."
assert np.isclose(auc_good5, auc_bad5) and ll_bad5 > ll_good5, "Same ranking, same AUC; calibration differs."
assert boot_ci5[0] < auc_mw5 < boot_ci5[1] and boot_ci5[1] - boot_ci5[0] > 0.05, "300 trials leave a visible uncertainty."
print("Self-check passed.")

**5.3 (4 points, written response).** (a) Explain why AUC equals $P(s_1 > s_0)$ in words, and why it ignores the base rate. (b) A decoder has AUC 0.97 and its "probabilities" have log loss 2.0. Is it a good decoder? (c) Single-neuron "choice probability" is the AUC between a neuron's responses on trials ending in choice A and in choice B. What value means "no relation", and why must it come with an interval?

**Your answer:**

*(replace this line with your answer)*

---
## Exercise 6 — Dispersion Checks with Error Bars (10 points)

Three simulated neurons, 60 trials each: a Poisson neuron, a neuron whose rate varies across trials (gamma–Poisson), and a regular neuron (binomial), all with mean count about 5.

In [ ]:
# 6.1 (6 points): For each neuron in cnt6, compute the Fano factor and its 95% bootstrap interval resampling trials (B = 3000, rng6), and the
# 95% range of Fano factors from Poisson neurons with the same mean and 60 trials (3000 simulations, rng6). Store results in
# out6 = {name: (ff, (boot_lo, boot_hi), (null_lo, null_hi))} and a verdict in verdict6 = {name: 'under' / 'consistent' / 'over'}, using
# 'over' if ff exceeds the null range, 'under' if below it.

rng6 = np.random.default_rng(6)
cnt6 = {'Poisson': np.random.default_rng(61).poisson(5, 60),
        'rate varies': np.random.default_rng(62).poisson(np.random.default_rng(63).gamma(3, 5 / 3, 60)),
        'regular': np.random.default_rng(64).binomial(9, 0.55, 60)}

# YOUR CODE HERE
out6 = verdict6 = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 6.1
assert out6 is not None and verdict6 is not None, "Set out6 and verdict6."
assert verdict6 == {'Poisson': 'consistent', 'rate varies': 'over', 'regular': 'under'}, "Check the three verdicts."
assert all(v[1][0] < v[0] < v[1][1] for v in out6.values()), "Each bootstrap interval contains its estimate."
print("Self-check passed.")

**6.2 (4 points, written response).** The bootstrap interval and the Poisson-null range answer different questions. State each question, and say which one a claim "this neuron is more variable than Poisson" needs.

**Your answer:**

*(replace this line with your answer)*

---
## Exercise 7 — Real Data: Error Bars for a Second Mouse (15 points)

**Data.** Session 25 of the Neuromatch Academy release of [Steinmetz et al. (2019)](https://doi.org/10.1038/s41586-019-1787-x) (mouse Radnitz), `data/steinmetz_s25_radnitz.npz`: counts 50–250 ms after onset, `counts7` (178, 1056); `area7` (1056,); contrasts `cl7`, `cr7` (178,).

In [ ]:
# PROVIDED: load session 25.
import os, urllib.request
DATA_FILE7 = 'steinmetz_s25_radnitz.npz'
for cand in [DATA_FILE7, os.path.join('..', '..', 'data', DATA_FILE7)]:
    if os.path.exists(cand):
        path7 = cand; break
else:
    urllib.request.urlretrieve('https://raw.githubusercontent.com/peter-adepoju/math-neuro-ml/8f9e0431a21d8ac0aa016e3502735f3b8627863a/data/' + DATA_FILE7, DATA_FILE7)
    path7 = DATA_FILE7
d7 = np.load(path7)
counts7 = d7['counts'].astype(float); area7 = d7['brain_area']
cl7, cr7 = d7['contrast_left'], d7['contrast_right']
X7 = counts7[:, area7 == 'VISp']; R7 = cr7 > 0                                 # (178, 162), (178,)
print(X7.shape, R7.sum(), "trials with a right grating")

In [ ]:
# 7.1 (7 points): Discrimination with error bars. Split trials with rng7 (first half of rng7.permutation(178) = training). On the test trials:
#   (a) AUC of the summed VISp count (auc_sum7) and of the naive-Bayes log-likelihood ratio with means fitted on training trials (pseudo-count 0.1)
#       (auc_nb7);
#   (b) 95% bootstrap intervals for both (ci_sum7, ci_nb7), and a PAIRED bootstrap interval for auc_nb7 - auc_sum7 (ci_diff7), resampling test
#       trials (B = 2000, rng7b below; skip resamples with only one class).

rng7 = np.random.default_rng(7)
rng7b = np.random.default_rng(71)

# YOUR CODE HERE
auc_sum7 = auc_nb7 = ci_sum7 = ci_nb7 = ci_diff7 = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 7.1
assert all(v is not None for v in (auc_sum7, auc_nb7, ci_sum7, ci_nb7, ci_diff7)), "Set all five results."
assert 0.85 < auc_sum7 < auc_nb7 <= 1 and ci_sum7[0] < auc_sum7 < ci_sum7[1], "Both decoders discriminate well."
assert ci_diff7[0] < auc_nb7 - auc_sum7 < ci_diff7[1], "The paired interval contains the observed difference."
print("Self-check passed.")

In [ ]:
# 7.2 (4 points): Signal and noise correlation. For the two highest-rate VISp neurons (pair7 = their indices within X7, by mean count over all
# trials), store the correlation of their counts pooled over all trials (r_pool7), on trials with cr7 == 1 (r_c1_7) and on trials with cr7 == 0
# (r_c0_7), each with a 95% bootstrap interval resampling trials (ci_pool7, ci_c1_7, ci_c0_7; B = 2000, rng7b; drop resamples in which a
# neuron has zero variance, which happens when it rarely fires). Also print, for each condition, on how many trials each neuron fired.

# YOUR CODE HERE
pair7 = r_pool7 = r_c1_7 = r_c0_7 = ci_pool7 = ci_c1_7 = ci_c0_7 = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 7.2
assert all(v is not None for v in (pair7, r_pool7, r_c1_7, r_c0_7, ci_pool7, ci_c1_7, ci_c0_7)), "Set all results."
assert r_pool7 > r_c1_7 and r_pool7 > r_c0_7 and r_pool7 > 0.5, "Pooling over the stimulus inflates the correlation."
assert abs(r_c0_7) < 0.2 and ci_c1_7[0] > 0, "Full contrast: a positive noise correlation; no grating: near zero (see the spike totals)."
print("Self-check passed.")

**7.3 (4 points, written response).** (a) Summarise 7.1 in one sentence a reviewer would accept, with intervals. (b) Interpret the three correlations of 7.2 in terms of signal and noise correlation, taking the interval widths into account. (c) Name one way the bootstrap intervals here could be too narrow.

**Your answer:**

*(replace this line with your answer)*

---
## Bonus Challenge — Is a Small AUC Real? (extra credit)

A neuron's "choice probability" is computed as the AUC between its counts on trials ending in a right turn and a left turn. With few trials, small departures from 0.5 occur by chance.
1. For the simulated neuron below (its counts do not depend on the choice), compute the AUC and a **permutation** null: shuffle the choice labels 5000 times (`np.random.default_rng(99)`), recompute the AUC each time, and report the two-sided p-value $P(|\mathrm{AUC}_{\text{perm}} - 0.5| \ge |\mathrm{AUC}_{\text{obs}} - 0.5|)$.
2. Repeat for a neuron whose mean count is 20% higher on right-turn trials.
3. Explain the difference between this permutation test and a bootstrap interval.

**Your code:**

In [ ]:
# YOUR CODE HERE (bonus, optional)
raise NotImplementedError

**Your written answer (bonus):**

*(replace this line with your answer)*

---
## Final Submission Checklist

- All `# YOUR CODE HERE` cells have working code (no `raise NotImplementedError` left in).
- All **Written response** cells contain your own answer, not the placeholder text.
- All self-check cells run without an `AssertionError`.
- All figures have a title, axis labels with units, a legend where appropriate, and no background grid.
- Every estimate you report has an interval, and every interval says how it was computed and what was resampled.
- Every correlation is accompanied by a plot or a check for nonlinearity, outliers and obvious confounders.
- You ran the whole notebook top to bottom at least once before submitting (Restart & Run All), and no cell raised an unexpected error.